<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/Quantum_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit-ibm-runtime
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as RuntimeEstimator
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=userdata.get("IBM_API_KEY"),
    instance=userdata.get("IBM_CRN"),
)

backend = service.least_busy(operational=True, simulator=False, min_num_qubits=16)
print(backend.name)

pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
estimator = RuntimeEstimator(mode=backend)

qiskit_runtime_service._discover_account:WARNING:2026-11-01 04:54:23,994: Loading account with the given token. A saved account will not be used.


ibm_fez


/tmp/ipykernel_2371/804663872.py:16: DeprecationWarning: The EstimatorV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_estimator.Estimator` provides a new Estimator implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Estimator` import will be updated to point to the client-side estimator in an upcoming release, with the legacy Estimator being available until the deprecation is completed.
  estimator = RuntimeEstimator(mode=backend)


In [2]:
!git clone https://github.com/suman-poola/Random-Forrest-s-p500.git

%cd Random-Forrest-s-p500

from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

Cloning into 'Random-Forrest-s-p500'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 136 (delta 77), reused 24 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 18.46 MiB | 13.22 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/Random-Forrest-s-p500


In [3]:
from Data_Frame_10y import build_dataset

df, x_test, x_train, y_train, y_test, encoding_table = build_dataset()

Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 101M/101M [00:00<00:00, 158MB/s] 


/content/Random-Forrest-s-p500/Data_Frame_10y.py:22: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="10y")
[*********************100%***********************]  503 of 503 completed
/content/Random-Forrest-s-p500/Data_Frame_10y.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['future return'] = df.groupby('ticker')['return'].shift(-1)


In [4]:
import json
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
!pip install qiskit
!pip install qiskit.circuit
!pip install qiskit.algorithms
!pip install qiskit.quantum_info
!pip install qiskit.primitives
!pip install qiskit_machine_learning
from qiskit import QuantumCircuit as QC
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import z_feature_map
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.utils import algorithm_globals
from qiskit_machine_learning.algorithms import NeuralNetworkRegressor
from qiskit_machine_learning.neural_networks import EstimatorQNN
from sklearn.model_selection import train_test_split

algorithm_globals.random_seed = 12345
estimator = Estimator()

ERROR: Could not find a version that satisfies the requirement qiskit.circuit (from versions: none)
ERROR: No matching distribution found for qiskit.circuit
ERROR: Could not find a version that satisfies the requirement qiskit.quantum_info (from versions: none)
ERROR: No matching distribution found for qiskit.quantum_info
ERROR: Could not find a version that satisfies the requirement qiskit.primitives (from versions: none)
ERROR: No matching distribution found for qiskit.primitives


In [5]:
from qiskit import QuantumCircuit as QC


In [6]:
def conv_circuit(params):
    target = QC(2)
    target.rz(-np.pi / 2, 1)
    target.cx(1, 0)
    target.rz(params[0], 0)
    target.ry(params[1], 1)
    target.cx(0, 1)
    target.ry(params[2], 1)
    target.cx(1, 0)
    target.rz(np.pi / 2, 0)
    return target

def conv_layer(num_qubits, param_prefix):
    qc = QC(num_qubits, name="Convolutional Layer")
    qubits = list(range(num_qubits))
    param_index = 0
    params = ParameterVector(param_prefix, length=num_qubits * 3)
    for q1, q2 in zip(qubits[0::2], qubits[1::2]):
        qc = qc.compose(conv_circuit(params[param_index : (param_index + 3)]), [q1, q2])
        qc.barrier()
        param_index += 3
    for q1, q2 in zip(qubits[1::2], qubits[2::2] + [0]):
        qc = qc.compose(conv_circuit(params[param_index : (param_index + 3)]), [q1, q2])
        qc.barrier()
        param_index += 3

    qc_inst = qc.to_instruction()

    qc = QC(num_qubits)
    qc.append(qc_inst, qubits)
    return qc

In [7]:
def pool_circuit(params):
    target = QC(2)
    target.rz(-np.pi / 2, 1)
    target.cx(1, 0)
    target.rz(params[0], 0)
    target.ry(params[1], 1)
    target.cx(0, 1)
    target.ry(params[2], 1)

    return target

def pool_layer(sources, sinks, param_prefix):
    num_qubits = len(sources) + len(sinks)
    qc = QC(num_qubits, name="Pooling Layer")
    param_index = 0
    params = ParameterVector(param_prefix, length=num_qubits // 2 * 3)
    for source, sink in zip(sources, sinks):
        qc = qc.compose(pool_circuit(params[param_index : (param_index + 3)]), [source, sink])
        qc.barrier()
        param_index += 3

    qc_inst = qc.to_instruction()

    qc = QC(num_qubits)
    qc.append(qc_inst, range(num_qubits))
    return qc


In [18]:
features = [ 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10', 'SMA 20', 'SMA 30', 'SMA 50', 'id','volume']
x_train_reduced = np.hstack([x_train_reduced, np.zeros((len(x_train_reduced), 1))])
x_test_reduced  = np.hstack([x_test_reduced,  np.zeros((len(x_test_reduced), 1))])

In [19]:
len(features)

15

In [20]:
x = df[features]
y = df['future return']

split_index = int(0.8 * len(df))
x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [21]:

feature_map = z_feature_map(16)

In [22]:
ansatz = QC(16, name="QCNN")

ansatz.compose(conv_layer(16,'c1'), list(range(16)), inplace=True)
ansatz.compose(pool_layer(list(range(0,8)), list(range(8,16)), "p1"), list(range(16)), inplace=True)

ansatz.compose(conv_layer(8,'c2'), list(range(8,16)), inplace=True)
ansatz.compose(pool_layer(list(range(0,4)), list(range(4,8)), "p2"), list(range(8,16)), inplace=True)

ansatz.compose(conv_layer(4,'c3'), list(range(12,16)), inplace=True)
ansatz.compose(pool_layer(list(range(0,2)), list(range(2,4)), "p3"), list(range(12,16)), inplace=True)

ansatz.compose(conv_layer(2,'c4'), list(range(14,16)), inplace=True)
ansatz.compose(pool_layer(list(range(0,1)), list(range(1,2)), "p4"), list(range(14,16)), inplace=True)

In [23]:
circuit = QC(16)
circuit.compose(feature_map, range(16), inplace=True)
circuit.compose(ansatz, range(16), inplace=True)

In [24]:
observable = SparsePauliOp.from_list([("Z" + "I" * 15, 1)])

In [25]:
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

qnn = EstimatorQNN(
    circuit=isa_circuit,
    observables=isa_observable,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
    estimator=estimator,
)

In [ ]:
regressor = NeuralNetworkRegressor(
    neural_network=qnn,
    optimizer=COBYLA(maxiter=30),
    loss="squared_error",
)

scale_factor = y_train.abs().max()
y_train_scaled = y_train / scale_factor

regressor.fit(x_train_reduced, y_train_scaled)